In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

train = pd.read_csv("/kaggle/input/mai-ml-lab-1-fiit-2025/train.csv")
test  = pd.read_csv("/kaggle/input/mai-ml-lab-1-fiit-2025/test.csv")

target_col = "RiskScore"

print("Train shape:", train.shape)
print("Test shape:", test.shape)


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/mai-ml-lab-1-fiit-2025/train.csv'

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(train[target_col], bins=50, kde=True)
plt.title("Распределение RiskScore")
plt.show()

plt.figure(figsize=(10,8))
corr = train.corr(numeric_only=True)
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Корреляции признаков")
plt.show()

sns.pairplot(train[["Age","AnnualIncome","CreditScore","LoanAmount","RiskScore"]])
plt.show()


In [ ]:
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def r2_score_custom(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - y_true.mean())**2)
    return 1 - ss_res/ss_tot if ss_tot != 0 else 0

def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100


In [ ]:
class ZScoreScaler:
    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0, ddof=0)
        self.std_[self.std_ == 0] = 1.0
        return self
    def transform(self, X):
        return (X - self.mean_) / self.std_
    def fit_transform(self, X):
        return self.fit(X).transform(X)

class MinMaxScaler:
    def fit(self, X):
        self.min_ = X.min(axis=0)
        self.max_ = X.max(axis=0)
        return self
    def transform(self, X):
        return (X - self.min_) / (self.max_ - self.min_ + 1e-8)
    def fit_transform(self, X):
        return self.fit(X).transform(X)


In [ ]:
class LinearRegressionCustom:
    def __init__(self, fit_method="normal", lr=1e-2, epochs=1000, batch_size=None,
                 l1=0.0, l2=0.0, random_state=42, verbose=False):
        self.fit_method = fit_method
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.l1 = l1
        self.l2 = l2
        self.random_state = random_state
        self.verbose = verbose

    def _add_bias(self, X):
        return np.hstack([np.ones((X.shape[0], 1)), X])

    def fit(self, X, y):
        Xb = self._add_bias(X)
        n, d = Xb.shape
        rng = np.random.default_rng(self.random_state)

        if self.fit_method == "normal":
            I = np.eye(d); I[0,0] = 0
            A = Xb.T @ Xb + self.l2 * I
            b = Xb.T @ y
            self.w_ = np.linalg.pinv(A) @ b
        elif self.fit_method in ("gd", "sgd"):
            self.w_ = rng.normal(0, 0.01, size=d)
            for epoch in range(self.epochs):
                if self.fit_method == "gd":
                    y_pred = Xb @ self.w_
                    grad = (2/n) * (Xb.T @ (y_pred - y))
                    l2_grad = 2 * self.l2 * self.w_; l2_grad[0] = 0
                    l1_grad = self.l1 * np.sign(self.w_); l1_grad[0] = 0
                    self.w_ -= self.lr * (grad + l2_grad + l1_grad)
                else:  # SGD
                    idx = rng.permutation(n)
                    bs = self.batch_size or 32
                    for start in range(0, n, bs):
                        batch_idx = idx[start:start+bs]
                        Xb_i = Xb[batch_idx]
                        y_i = y[batch_idx]
                        y_pred = Xb_i @ self.w_
                        grad = (2/len(batch_idx)) * (Xb_i.T @ (y_pred - y_i))
                        l2_grad = 2 * self.l2 * self.w_; l2_grad[0] = 0
                        l1_grad = self.l1 * np.sign(self.w_); l1_grad[0] = 0
                        self.w_ -= self.lr * (grad + l2_grad + l1_grad)
        else:
            raise ValueError("Unknown fit_method")
        return self

    def predict(self, X):
        Xb = self._add_bias(X)
        return Xb @ self.w_


In [ ]:
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove(target_col)
if "ID" in numeric_cols: numeric_cols.remove("ID")

categorical_cols = [c for c in train.columns if train[c].dtype == "object"]

train_num = train[numeric_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
test_num  = test[numeric_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

train_cat = pd.get_dummies(train[categorical_cols], drop_first=True)
test_cat  = pd.get_dummies(test[categorical_cols], drop_first=True)

train_enc = pd.concat([train_num, train_cat], axis=1)
test_enc  = pd.concat([test_num,  test_cat], axis=1)

train_enc, test_enc = train_enc.align(test_enc, join="left", axis=1, fill_value=0)

X_train = train_enc.values.astype(float)
X_test  = test_enc.values.astype(float)
y_train = train[target_col].replace([np.inf, -np.inf], np.nan).fillna(0).values.astype(float)

mask = ~np.isnan(y_train)
X_train = X_train[mask]
y_train = y_train[mask]


In [ ]:
scaler = ZScoreScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

model = LinearRegressionCustom(fit_method="normal", l2=10000).fit(X_train_s, y_train)
test_pred = model.predict(X_test_s)

submission = pd.DataFrame({
    "ID": test["ID"],
    "RiskScore": test_pred
})
submission.to_csv("/kaggle/working/submission.csv", index=False)
print("Saved submission.csv")
print(submission.head())

y_pred_train = model.predict(X_train_s)
print("Train MSE:", mse(y_train, y_pred_train))
print("Train MAE:", mae(y_train, y_pred_train))
print("Train R2:", r2_score_custom(y_train, y_pred_train))
print("Train MAPE:", mape(y_train, y_pred_train))


In [ ]:
from sklearn.model_selection import KFold, LeaveOneOut

def cross_val_score_custom(model_class, X, y, cv=5, fit_method="normal"):
    kf = KFold(n_splits=cv, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        model = model_class(fit_method=fit_method).fit(X_tr, y_tr)
        y_pred = model.predict(X_val)
        scores.append(mse(y_val, y_pred))
    return np.mean(scores)

print("5-fold CV MSE:", cross_val_score_custom)